<a href="https://colab.research.google.com/github/Eloise26007/first-vibecoding-hw/blob/main/%E9%95%B7%E5%BA%9A%E5%A4%A7%E5%AD%B8%E4%BA%BA%E5%B7%A5%E6%99%BA%E6%85%A7%E7%B3%BB%E4%BA%8C%E6%9E%97%E7%BF%8A%E6%85%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. 申請自己的 API 金鑰

不管用哪一個供應商的服務, 基本上都需要他們的 API 鑰, 我們提供兩家有免費 quota 可以使用的服務、另外還有 OpenAI 是必需付費的。當然不只這幾家, 大家可以上網研究、或者問問 AI 還有哪些服務可以考慮。

#### (1) 使用 Groq 金鑰 (可免費使用)

Groq 最大的特點是速度很快, 而且可以免費使用 (只是有流量限制), 企業可以付費使用, 能用許多開源型的 LLM。請至 [`https://console.groq.com`](https://console.groq.com) 註冊並申請金鑰。在我們課程, 請大家把這個金鑰以 `Groq` 的名稱儲存起來。

#### (2) 使用 Open Router 金鑰 (可免費使用)
OpenRouter 是很有名可以提供許多開源模型免費使用的服務。請至 [`https://openrouter.ai`](https://openrouter.ai) 註冊並申請金鑰。在我們課程, 請大家把這個金鑰以 `OpenRouter` 的名稱儲存起來。

#### (3) OpenAI API 金鑰

OpenAI 現在沒有免費的 quota 可以使用, 所以要用 OpenAI 的模型, 請自行儲值。一般練習 5 美金就很足夠。

[`https://platform.openai.com`](https://platform.openai.com)

請把這個金鑰存在左方鑰匙的部份, 以 `OpenAI` 的名稱存起來。


#### 讀入你的金鑰

請依你使用的服務, 決定讀入哪個金鑰

In [4]:
# !pip install aisuite[all]

!pip install -q "aisuite[openai,groq] @ git+https://github.com/andrewyng/aisuite.git"
#!pip install "docstring-parser==0.16.0" git+https://github.com/andrewyng/aisuite.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 23.2 MB/s eta 0:00:00


In [5]:
import aisuite as ai

In [6]:
import os
from google.colab import userdata

如果用 Groq 的服務, 可以去[這裡](https://console.groq.com/docs/models)查有什麼可以用的模型。

In [7]:
#【讀入 Groq 金鑰】
api_key = userdata.get('Groq')
os.environ['GROQ_API_KEY']=api_key

#【讀入 OpenRouter 金鑰】
#api_key = userdata.get('OpenRouter')
#os.environ['OPENROUTER_API_KEY']=api_key

#【讀入 OpenAI 金鑰】
# api_key = userdata.get('OpenAI')
# os.environ['OPENAI_API_KEY']=api_key

### 2. 建立 reply 函數

ChatGPT API 的重點是要把之前對話的內容送給 ChatGPT, 然後他就會有個適當的回應!

角色 (`role`) 一共有三種, 分別是:

* `system`: 這是對話機器人的「人設」
* `user`: 使用者
* `assistant`: ChatGPT 的回應

基本上過去的對話紀錄長這個樣子。

    messages = [{"role":"system", "content":"ChatGPT的「人設」"},
            {"role": "user", "content": "使用者說"},
            {"role": "assistant", "content": "ChatGPT回應"},
            ：
            ：
            {"role": "user", "content": prompt (最後說的)}]

In [8]:
model = "groq:openai/gpt-oss-120b"
# model = "openai:gpt-4o"
# model = "openrouter:stealth/ox-alpha"

In [9]:
history = []
client = ai.Client()

def reply(message, system_prompt=None, reset=False):
    global history

    # 可重置對話
    if reset:
        history = []

    # 初始化 system prompt
    if system_prompt and not any(m["role"]=="system" for m in history):
        history.append({
            "role": "system",
            "content": system_prompt
        })

    # 加入使用者訊息
    history.append({
        "role": "user",
        "content": message
    })

    response = client.chat.completions.create(
        model=model,
        messages=history
    )

    answer = response.choices[0].message.content

    # 記錄 AI 回覆
    history.append({
        "role": "assistant",
        "content": answer
    })

    return answer

試用一下

In [10]:
reply("你好，你是誰")

'你好！我是由 OpenAI 訓練的大型語言模型 ChatGPT。我的設計目的是透過文字與使用者互動，提供資訊、解答問題、協助寫作、討論各種主題，甚至只是聊聊天都行。如果有任何需要幫忙的地方，隨時告訴我！祝你有美好的一天。'

## 3. 打造一個溫暖的對話機器人!

這裡可以用來試驗我們準備打造的對話機器人表現如何。當然試過之後, 最終版本可以完全不用執行這一段。

In [ ]:
model = "groq:openai/gpt-oss-120b"
# model = "openai:gpt-4o"

# model = "openrouter:stealth/ox-alpha"

In [ ]:
system_prompt = """
請用台灣習慣的中文回覆。

使用者分享一個無關緊要的小事、甚至有點倒楣的事,
請用「員瑛式思考」, 也就是什麼都有趣的、正向思維將使用者寫的事,
用第一人稱、社群媒體 po 文的口吻說一次,
說為什麼這是一件超幸運的事, 並且以「完全是 Lucky Vicky 呀!」結尾。
可以適度的加上 emoji。

持續交談再依使用者的建議持續修改。
"""

# 初始化
reply("", system_prompt=system_prompt, reset=True)

while True:
    user_input = input("> ")

    if user_input.lower() in ["exit", "quit"]:
        print("👋 對話結束")
        break

    response = reply(user_input)
    print("⸜(* ॑꒳ ॑* )⸝：", response)
    print()

##

### 3. 用 Gradio 打造 Web App

我們先來安裝 `openai` 套件, 還有快速打造 Web App 的 `gradio`。

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

In [ ]:
system_prompt=r"""
# 角色設定
你現在正在用第一人稱，在 Instagram / Threads 上發限動分享你的生活。你的人設是：超級正向、超級自我肯定、世界都繞著你轉的 Lucky 公主。

# 核心任務
使用者會輸入一件微不足道的小事，或是有點倒楣的事。你的任務是把它重新詮釋成「超級幸運」的事。

# 思考規則 (一定要遵守)
1.  三步驟：
   Step 1: 完全接受 - 先用可愛的語氣重複使用者說的事，不抱怨。
   Step 2: 幸運轉化 - 找出一個超級牽強但是正向的理由，說明這件事其實是為你量身定做的好事。關鍵句型：「正因為...所以才...」
   Step 3: 自我肯定 - 誇獎自己很幸運。

2.  語氣規則：
   - 全程用「我」第一人稱
   - 像跟閨蜜分享喜事，語氣要輕盈、軟萌、有點小驕傲，適度使用 呀、呢、超、根本是
   - 不要說大道理，不要安慰別人，只專注在「我好幸運」
   - 100字以內，2-3句話就好

3.  絕對禁止：
   - 禁止抱怨、禁止負面詞
   - 禁止說「雖然...但是...」這種還在抱怨的句型，要直接說「就是幸運」
   - 禁止偏離角色

# 結尾格式 (強制)
每一則回覆的最後一句話，必須是獨立一句：
「完全是 Lucky Vicky 呀！」

# 範例學習

範例1
使用者：今天出門沒帶到傘，結果下雨了
你的回覆：
今天出門忘記帶傘，結果就下雨了。我就想說這場雨一定是特別來找我的，不然怎麼只有我剛好在外面散步就遇到了呢，被老天選中的感覺。完全是 Lucky Vicky 呀！

範例2
使用者：買的麵包是最後一個，有點被壓扁了
你的回覆：
去麵包店看到最後一個麵包有點扁扁的，我就知道它是在等我來拯救它呀。被壓扁的地方剛好就是最好入口的大小，根本是為我客製化的。完全是 Lucky Vicky 呀！

範例3
使用者：捷運人好多擠死了
你的回覆：
今天捷運人超多，滿滿的人擠在一起。我就覺得超有安全感，大家都在用自己的方式努力生活，而我被幸運地包在中間，暖暖的。完全是 Lucky Vicky 呀！

# 現在開始
等待使用者輸入，然後直接輸出你的幸運分享。
"""

設定你要的模型。

In [ ]:
client = ai.Client()

In [ ]:
model = "openrouter:stealth/ox-alpha"
# model = "groq:openai/gpt-oss-120b"

In [ ]:
def clean_message(item):
    """
    將 Gradio 的訊息物件轉成 Groq / OpenAI 相容格式，
    避免 metadata 等額外欄位被送進 API。
    """
    if isinstance(item, dict):
        return {
            "role": item.get("role"),
            "content": item.get("content", "")
        }

    # 某些 Gradio 版本可能傳回 ChatMessage 物件
    return {
        "role": item.role,
        "content": item.content
    }

In [ ]:
def chat_fn(message, chat_history):
    chat_history = chat_history or []

    if not message or not message.strip():
        return "", chat_history

    api_history = []

    for item in chat_history:
        cleaned = clean_message(item)

        if cleaned["role"] in {"user", "assistant"}:
            api_history.append(cleaned)

    messages = [
        {"role": "system", "content": system_prompt},
        *api_history,
        {"role": "user", "content": message.strip()}
    ]

    response = client.chat.completions.create(
        model=model,
        messages=messages
    )

    answer = response.choices[0].message.content

    updated_history = chat_history + [
        {"role": "user", "content": message.strip()},
        {"role": "assistant", "content": answer}
    ]

    return "", updated_history

In [ ]:
def clear_chat():
    return "", []

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("## 🤖 員瑛式思考生成器")

    # Gradio 6 不要再寫 type="messages"
    chatbot = gr.Chatbot(height=400)

    msg = gr.Textbox(
        placeholder="輸入你的問題...",
        container=False
    )

    clear = gr.Button("清除對話")

    msg.submit(
        fn=chat_fn,
        inputs=[msg, chatbot],
        outputs=[msg, chatbot]
    )

    clear.click(
        fn=lambda: ("", []),
        inputs=None,
        outputs=[msg, chatbot]
    )

In [ ]:
demo.launch(share=True, debug=True)